# 03a: Getting the data ready for machine learning

**Goal:** turn the cleaned GLP-1 reports into **one table the models can use directly**:
one row per report, the answer column `is_serious`, **no missing values**, and **nothing that gives away the answer**.

| In | Out |
|---|---|
| `data/02_intermediate/glp1_reports.parquet` (one row per report) | `data/03_model/glp1_model_ready.parquet` |
| `data/02_intermediate/glp1_reactions.parquet` (one row per side effect) | `reports/ml_prep_decisions.csv` (why each column was kept or dropped) |
| `src/reaction_groups.csv` (side effect → group) | `reports/ml_feature_dictionary.csv` (what each model column means) |

This follows the points agreed in the team chat (Brian's review + our cleaning decisions).
Every decision is in the decision table in Step 3, so it can go straight into the report.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
IN_DIR = PROJECT_ROOT / "data" / "02_intermediate"
OUT_DIR = PROJECT_ROOT / "data" / "03_model"
REPORT_DIR = PROJECT_ROOT / "reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

SUFFIX = ""          # "" = full 2022-2026 data
MISSING_LIMIT = 30   # Lecture 4: more than 30% missing -> consider dropping the column

reports = pd.read_parquet(IN_DIR / f"glp1_reports{SUFFIX}.parquet")
reactions = pd.read_parquet(IN_DIR / f"glp1_reactions{SUFFIX}.parquet")
group_map = pd.read_csv(PROJECT_ROOT / "src" / "reaction_groups.csv")

print(f"Reports:   {reports.shape[0]:,} rows x {reports.shape[1]} columns")
print(f"Reactions: {reactions.shape[0]:,} rows (side-effect mentions)")
print(f"Reaction terms with a group: {len(group_map):,}")

## Step 1: Check the starting point

Before changing anything, confirm three things:
1. **One row per report**: no report appears twice.
2. **One version per case**: in cleaning we kept only the latest version of each case, so each `caseid` should appear once.
   This is why a normal stratified train/test split is safe (the same case cannot end up in both train and test).
3. **How many reports are serious**: this is the share the model has to find.

In [ ]:
n = len(reports)
print(f"Reports:                 {n:,}")
print(f"Unique primaryid:        {reports['primaryid'].nunique():,}")
print(f"Unique caseid:           {reports['caseid'].nunique():,}")
print(f"Serious reports:         {reports['is_serious'].sum():,} ({reports['is_serious'].mean()*100:.1f}%)")

assert reports["primaryid"].is_unique, "Some reports appear twice: check the cleaning step"
if not reports["caseid"].is_unique:
    print("WARNING: some cases appear more than once -> use a grouped split by caseid")
else:
    print("OK: every case appears once -> a stratified 80/20 split is safe")

## Step 2: How much is missing in each column?

Some models (logistic regression, random forest in scikit-learn) **cannot work with empty cells**,
so every column must be filled, turned into a flag, or dropped.
The table shows the share of reports where each column is empty **or** says "Unknown".

In [ ]:
def missing_share(s):
    empty = s.isna()
    if s.dtype == object or str(s.dtype) == "category":
        empty = empty | s.astype(str).str.strip().str.upper().isin(["UNKNOWN", "UNK", "NAN", "NONE", ""])
    return round(empty.mean() * 100, 1)

missing = (pd.DataFrame({
               "column": reports.columns,
               "type": [str(t) for t in reports.dtypes],
               "missing_or_unknown_%": [missing_share(reports[c]) for c in reports.columns],
               "distinct_values": [reports[c].nunique() for c in reports.columns]})
           .sort_values("missing_or_unknown_%", ascending=False)
           .reset_index(drop=True))
missing["over_30%"] = np.where(missing["missing_or_unknown_%"] > MISSING_LIMIT, "yes", "")
missing

In [ ]:
def pct(col):
    return float(missing.loc[missing["column"] == col, "missing_or_unknown_%"].iloc[0]) if col in reports else float("nan")

dose_missing = pct("dose_amt")
weight_missing = pct("weight_kg")
age_missing = pct("age_years")
print(f"Dose missing:   {dose_missing}%")
print(f"Weight missing: {weight_missing}%")
print(f"Age missing:    {age_missing}%")

## Step 3: Decision for every column

The rule of thumb: **keep a column only if it is known when the report arrives, is not just an ID,
does not repeat another column, and does not give away the answer.**

- **Leakage** = a column that *is* the answer in disguise (for example "died" or "hospitalised").
  It is like giving students the answer key during the exam: the model looks perfect but has learned nothing useful.
- **Missing values** are never guessed (no average-filling). A missing category becomes its own group, **"Unknown"**,
  because "the reporter did not say" can itself be a signal.

In [ ]:
dose_rule = "Drop the number, keep a yes/no flag"

decisions = pd.DataFrame([
    # column(s), decision, reason
    ("primaryid, caseid", "Keep in file, NOT a feature",
     "They are IDs, not information about the report. Kept for joining and duplicate checks."),
    ("is_serious", "Target (the answer)",
     "1 = death, life-threatening, hospitalisation, disability, birth defect or required intervention."),
    ("is_death, is_life_threatening, is_hospitalized, is_disability", "Drop (leakage)",
     "These define is_serious, so they give away the answer."),
    ("'Outcome words (death, hospital)' reaction group", "Drop (leakage)",
     "Side-effect terms such as 'death' or 'hospitalisation' are outcomes, not side effects."),
    ("drug", "Keep (category)", "The main comparison of the project: the four GLP-1 ingredients."),
    ("brand, drugname, prod_ai", "Drop",
     "Repeat the drug in more detail (free text / brand names); brand also has gaps."),
    ("route", "Keep, simplified to Injection / Oral / Unknown",
     "Raw text has many spellings; the useful part is pill vs injection."),
    ("dose_form", "Drop", "Repeats route (tablet vs solution)."),
    ("dose_amt, dose_unit", f"{dose_rule} -> dose_reported",
     f"{dose_missing}% missing, and mg is not comparable across drugs (1 mg weekly injection vs 14 mg daily pill are both normal doses)."),
    ("is_compounded", "Drop", "Every report is 0 (no compounded product found), so it carries no information."),
    ("age_years, age_raw, age_cod", "Drop", f"{age_missing}% missing; age_group already covers age."),
    ("age_group", "Keep (category incl. Unknown)", "Standard age bands; no guessing of missing ages."),
    ("weight_kg", "Drop (no flag either)",
     f"{weight_missing}% missing. The few reports with a weight are mostly serious ones, because serious cases get "
     "follow-up questions. A weight being filled in is a result of being serious, not a cause (see Step 5)."),
    ("sex", "Keep (F / M / Unknown)", "Basic patient information."),
    ("reporter_type", "Keep (category incl. Unknown)", "Who filed the report (doctor, pharmacist, consumer, lawyer)."),
    ("is_hcp", "Drop", "Repeats reporter_type (health professional yes/no)."),
    ("reporter_country", "Drop",
     "FDA rules mainly require drug companies to send reports from outside the US when they are serious, "
     "so 'outside US' almost means 'serious' (see Step 5). It would teach the model a reporting rule, not a medical pattern."),
    ("indication_group", "Keep (category incl. Unknown)",
     "Why the drug was taken (diabetes, weight loss, other). About half are Unknown; kept as its own group, not guessed."),
    ("n_reactions, n_other_drugs", "Keep (numbers)",
     "How many side effects and other medicines are listed. Skewed (a few reports list over 50); tree models do not mind, scale them for logistic regression."),
    ("year", "Keep (number)", "Reporting changes over time (new drugs, more weight-loss use)."),
    ("quarter", "Drop", "Quarter is when the FDA received the report, not a real season of illness."),
    ("year_quarter", "Drop", "Repeats year + quarter."),
    ("reaction groups (from the reactions table)", "Keep as 0/1 columns (rg_...)",
     "One column per side-effect group: 1 if the report mentions it. Yes/no can never be missing."),
], columns=["Column / issue", "Decision", "Reason"])

decisions.to_csv(REPORT_DIR / "ml_prep_decisions.csv", index=False)
decisions.set_index("Column / issue")

## Step 4: Build the model table

### 4a. Report-level columns

In [ ]:
r = reports.copy()
model = pd.DataFrame({"primaryid": r["primaryid"], "caseid": r["caseid"]})

# Categories: anything empty becomes "Unknown"
for col in ["drug", "age_group", "sex", "reporter_type", "indication_group"]:
    model[col] = r[col].astype("string").fillna("Unknown").replace({"": "Unknown"})

# Route -> Injection / Oral / Unknown
route = r["route"].astype("string").fillna("").str.upper()
model["route_group"] = np.select(
    [route.str.contains("SUBCUT|INJECT|INTRAMUSC|INTRAVEN"), route.str.contains("ORAL")],
    ["Injection", "Oral"], default="Unknown")

# Yes/no flag instead of a dose number with gaps (mg is not comparable across drugs)
model["dose_reported"] = r["dose_amt"].notna().astype(int)

# Numbers
model["year"] = r["year"].astype(int)
model["n_reactions"] = r["n_reactions"].fillna(0).astype(int)
model["n_other_drugs"] = r["n_other_drugs"].fillna(0).astype(int)

# Target
model["is_serious"] = r["is_serious"].astype(int)

print(model["route_group"].value_counts().to_string())

### 4b. Side-effect groups as yes/no columns

Each report can list many side effects. For the model, each **group** becomes one column:
1 if the report mentions any side effect in that group, 0 if not.
The **"Outcome words"** group is left out (leakage, see Step 3).

In [ ]:
mapping = (group_map[["pt", "reaction_group"]]
           .assign(pt=lambda d: d["pt"].astype(str).str.strip().str.upper())
           .drop_duplicates("pt"))

reac = reactions[["primaryid", "pt"]].copy()
reac["pt"] = reac["pt"].astype(str).str.strip().str.upper()
reac = reac.merge(mapping, on="pt", how="left")
reac["reaction_group"] = reac["reaction_group"].fillna("Other")

LEAK_GROUPS = [g for g in reac["reaction_group"].unique() if g.lower().startswith("outcome")]
print("Left out (leakage):", LEAK_GROUPS)
reac = reac[~reac["reaction_group"].isin(LEAK_GROUPS)]

def slug(name):
    return "rg_" + re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

flags = (reac.drop_duplicates(["primaryid", "reaction_group"])
             .assign(v=1)
             .pivot_table(index="primaryid", columns="reaction_group", values="v",
                          aggfunc="max", fill_value=0))
group_names = {slug(g): g for g in flags.columns}
flags.columns = [slug(c) for c in flags.columns]

model = model.merge(flags, left_on="primaryid", right_index=True, how="left")
RG_COLS = list(flags.columns)
model[RG_COLS] = model[RG_COLS].fillna(0).astype(int)

(model[RG_COLS].mean().mul(100).round(1)
     .rename(index=group_names).sort_values(ascending=False)
     .rename("% of reports mentioning it").to_frame())

## Step 5: Leakage check

If one single column almost perfectly predicts "serious", it is probably the answer in disguise.
For every yes/no column we look at **% serious when the column = 1**. Anything above 90% is flagged.
(The overall rate is about 11%.)

A column can also leak **more quietly**: through the *rules of reporting* rather than medicine.
The table below shows the two columns we dropped for this reason:

- **Country:** FDA rules mainly require drug companies to send reports from outside the US when they are serious,
  so reports from outside the US are far more often serious.
- **Weight filled in:** serious cases get follow-up questions, so their forms are more complete.

Both are real patterns, but they describe *how reports are filed*, not *what makes a side effect serious*,
so they would make the model look better without teaching us anything useful.

In [ ]:
binary_cols = [c for c in model.columns
               if c not in ("primaryid", "caseid", "is_serious") and set(model[c].unique()) <= {0, 1}]
rows = []
for c in binary_cols:
    on = model[c] == 1
    if on.sum() >= 100:
        rows.append((c, int(on.sum()), round(model.loc[on, "is_serious"].mean() * 100, 1)))
leak_check = pd.DataFrame(rows, columns=["column", "reports_with_1", "% serious when 1"]) \
               .sort_values("% serious when 1", ascending=False).reset_index(drop=True)

suspicious = leak_check[leak_check["% serious when 1"] > 90]
print("Columns flagged (> 90% serious):", "none" if suspicious.empty else list(suspicious["column"]))

# The two reporting-rule columns we dropped, shown with the evidence
country = r["reporter_country"].astype(str).str.upper().isin(["US", "USA", "UNITED STATES"])
weight = r["weight_kg"].notna() if "weight_kg" in r else pd.Series(False, index=r.index)
dropped = pd.DataFrame({
    "% serious": [r.loc[country, "is_serious"].mean(), r.loc[~country, "is_serious"].mean(),
                  r.loc[weight, "is_serious"].mean(), r.loc[~weight, "is_serious"].mean()],
    "reports": [country.sum(), (~country).sum(), weight.sum(), (~weight).sum()]},
    index=["Reported in the US", "Reported outside the US", "Weight filled in", "Weight not filled in"])
dropped["% serious"] = (dropped["% serious"] * 100).round(1)
print("\nDropped as reporting rules (not medicine):")
print(dropped.to_string())
leak_check.head(10)

## Step 6: Final checks

In [ ]:
LEAKAGE_COLS = ["is_death", "is_life_threatening", "is_hospitalized", "is_disability",
                "DE", "LT", "HO", "DS", "CA", "RI", "OT"]
ID_COLS = ["primaryid", "caseid"]
TARGET = "is_serious"
FEATURES = [c for c in model.columns if c not in ID_COLS + [TARGET]]

assert model["primaryid"].is_unique, "duplicate reports"
assert len(model) == len(reports), "rows were lost or added"
assert model.isna().sum().sum() == 0, "there are still missing values"
assert not set(LEAKAGE_COLS) & set(FEATURES), "a leakage column is still a feature"
assert "country_group" not in FEATURES and "weight_reported" not in FEATURES
assert not any(c.startswith("rg_outcome") for c in FEATURES), "outcome-words group is still a feature"
assert abs(model[TARGET].mean() - reports["is_serious"].mean()) < 1e-9, "target changed"

CAT_COLS = [c for c in FEATURES if model[c].dtype == "string" or model[c].dtype == object]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

print(f"Rows:      {len(model):,}  (same as before)")
print(f"Features:  {len(FEATURES)}  ({len(CAT_COLS)} categories, {len(NUM_COLS)} numbers / yes-no)")
print(f"Missing values: {int(model.isna().sum().sum())}")
print(f"Serious:   {model[TARGET].mean()*100:.1f}%")
print("\nCategory columns:", CAT_COLS)

### Before vs after (for the report)

In [ ]:
summary = pd.DataFrame({
    "": ["Rows (reports)", "Columns", "Columns usable as model features", "Cells with missing values",
         "Leakage columns", "Serious reports"],
    "Before (cleaned reports)": [f"{len(reports):,}", reports.shape[1], "not decided",
                                 f"{int(reports.isna().sum().sum()):,}",
                                 sum(c in reports for c in LEAKAGE_COLS), f"{reports['is_serious'].mean()*100:.1f}%"],
    "After (model-ready)": [f"{len(model):,}", model.shape[1], len(FEATURES), 0, 0,
                            f"{model[TARGET].mean()*100:.1f}%"],
})
summary.set_index("")

## Step 7: Save

In [ ]:
for c in CAT_COLS:
    model[c] = model[c].astype(str)

model.to_parquet(OUT_DIR / f"glp1_model_ready{SUFFIX}.parquet", index=False)

meaning = {
    "drug": "GLP-1 ingredient (semaglutide, tirzepatide, dulaglutide, liraglutide)",
    "age_group": "Age band, or Unknown", "sex": "F / M / Unknown",
    "reporter_type": "Who filed the report", "indication_group": "Why the drug was taken",
    "route_group": "Injection / Oral / Unknown", "country_group": "US / Outside US / Unknown",
    "dose_reported": "1 = the report gives a dose", "weight_reported": "1 = the report gives a weight",
    "is_compounded": "1 = compounded (pharmacy-made) product",
    "year": "Year the FDA received the report", "n_reactions": "Number of side effects listed",
    "n_other_drugs": "Number of other medicines listed",
}
meaning.update({c: f"1 = report mentions a side effect in '{group_names[c]}'" for c in RG_COLS})
dictionary = pd.DataFrame({"feature": FEATURES,
                           "type": ["category" if c in CAT_COLS else "number / 0-1" for c in FEATURES],
                           "meaning": [meaning.get(c, "") for c in FEATURES]})
dictionary.to_csv(REPORT_DIR / "ml_feature_dictionary.csv", index=False)

print("saved ->", OUT_DIR / f"glp1_model_ready{SUFFIX}.parquet")
print("saved ->", REPORT_DIR / "ml_prep_decisions.csv")
print("saved ->", REPORT_DIR / "ml_feature_dictionary.csv")

## How to use this file in the model notebook

```python
df = pd.read_parquet("../data/03_model/glp1_model_ready.parquet")
ID_COLS, TARGET = ["primaryid", "caseid"], "is_serious"
X, y = df.drop(columns=ID_COLS + [TARGET]), df[TARGET]
```

1. **Split first** (stratified 80/20, `random_state=42`), so the 11% serious share is the same in train and test.
2. **One-hot encode the category columns and scale the numbers inside a `Pipeline`**, fitted on the training data only
   (fitting on all data would let the test set leak into training).
3. Handle the imbalance with `class_weight="balanced"` (or SMOTE on the training data only).
4. Judge models by recall, F1 and PR-AUC, not accuracy (always guessing "not serious" already scores about 89%).